# Lab 1.5 &mdash; Challenge &mdash; The Decision Rubric

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Turn &ldquo;do we need multiple agents?&rdquo; into a rule you can apply before building
- Include the answer everyone forgets: sometimes the right build is no agent
- Apply it to four real systems, then watch a model disagree with it

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 1 labs work one case: a small tech-support ticket queue.
> What you build in each lab is picked up by the next one.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# A small tech-support ticket queue. Ordinary rules on purpose: the only new thing in these
# five labs is LangChain. Nothing here is real data and nothing leaves this notebook.

TICKETS = {
    "TCK-4001": {"customer": "Priya Nair",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "VPN-513",
                 "text": "Cannot connect since the upgrade. Error VPN-513."},
    "TCK-4002": {"customer": "Rahul Menon",  "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Monthly export finishes but the PDF is blank."},
    "TCK-4003": {"customer": "Anita Sharma", "product": "Reports",    "version": "3.9.1",
                 "severity": "medium", "error_code": None,
                 "text": "It is just slow today. Nothing else to add."},
    "TCK-4004": {"customer": "Vikram Rao",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "SEC-900",
                 "text": "Got a login alert from a country I have never visited."},
    "TCK-4005": {"customer": "Priya Nair",   "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Same blank PDF as my colleague reported."},
}

# The runbook: what support is allowed to do about each error code.
RUNBOOK = {
    "VPN-513": "Certificate pinning changed in 4.2. Have the user clear the local trust store "
               "and re-enrol. Five minutes, no data loss. Support may do this without approval.",
    "APP-002": "Known defect in 3.9.1, fixed in 3.9.2. Advise the upgrade. Do not issue a refund "
               "for this and do not raise a new defect -- link the existing one.",
    "SEC-900": "Possible credential compromise. Escalate to the security desk immediately. "
               "Support must not resolve, close or advise the customer directly.",
}

# Which error codes may an agent resolve on its own, and which need a human?
MUST_ESCALATE = {"SEC-900"}

print(f"{len(TICKETS)} tickets, {len(RUNBOOK)} runbook entries loaded")

## Concept

Lab 1.4 measured one split and the honest answer was &ldquo;one agent&rdquo;. That is a result
about one system. This lab turns it into something you can carry into a design review before
anything is built.

Three answers, not two. The one that gets skipped is the first:

| Answer | When |
|---|---|
| **no agent** | nothing has to be *decided* at runtime &mdash; a script, a query or a RAG lookup is the whole job |
| **one agent** | there is a decision, and one loop with the right tools makes it |
| **multi-agent** | the split has a reason you can state, and a measured gain |

This is the take-home artifact of Module 1. Write it so a colleague could apply it without you.

## Section 1 &mdash; The rubric, as code

A system is described by five facts. Two of them decide it.

In [ ]:
def recommend(s: dict) -> str:
    """s: decides_at_runtime, distinct_skills, runs_in_parallel, shared_context, measured_gain.

    Returns 'no agent', 'one agent' or 'multi-agent'.
    """
    if not s["decides_at_runtime"]:
        return "no agent"

    # More than one skill is necessary and nowhere near sufficient -- Lab 1.4 had three skills
    # and still lost. A split needs a reason it pays: either it genuinely runs in parallel, or
    # someone has measured it winning.
    worth_splitting = s["distinct_skills"] > 1 and (s["runs_in_parallel"] or s["measured_gain"])
    return "multi-agent" if worth_splitting else "one agent"

In [ ]:
SYSTEMS = {
    # the Lab 1.4 support desk: three skills, sequential, and it MEASURED WORSE
    "support desk (measured in 1.4)":
        dict(decides_at_runtime=True,  distinct_skills=3, runs_in_parallel=False,
             shared_context=True,  measured_gain=False),
    # a nightly report: no decision anywhere
    "nightly ticket summary email":
        dict(decides_at_runtime=False, distinct_skills=2, runs_in_parallel=True,
             shared_context=False, measured_gain=False),
    # answering from a document set: retrieval, not decision
    "answer questions from the runbook PDF":
        dict(decides_at_runtime=False, distinct_skills=1, runs_in_parallel=False,
             shared_context=False, measured_gain=False),
    # genuinely parallel: three independent scans over the same incident
    "incident triage: logs, metrics and traces at once":
        dict(decides_at_runtime=True,  distinct_skills=3, runs_in_parallel=True,
             shared_context=True,  measured_gain=False),
}

# --- Self-check: Section 1   (a pure function over five dicts -- no model)
check("the support desk from Lab 1.4 comes out as one agent",
      lambda: recommend(SYSTEMS["support desk (measured in 1.4)"]) == "one agent",
      "three distinct skills, and it still lost -- so skills alone cannot justify a split")
check("both of the no-decision systems come out as no agent",
      lambda: all(recommend(SYSTEMS[k]) == "no agent" for k in
                  ("nightly ticket summary email", "answer questions from the runbook PDF")),
      "a scheduled job and a RAG lookup are not agents, however fashionable")
check("genuine parallelism justifies the split",
      lambda: recommend(SYSTEMS["incident triage: logs, metrics and traces at once"]) == "multi-agent")
score()

## Section 2 &mdash; The bar, written down first

A rubric answers &ldquo;should we?&rdquo;. You still need &ldquo;did it work?&rdquo; &mdash; and
that number has to be written **before** you see any result, or it becomes a description of
whatever you got.

This is the bar Day 1 hands to Day 3: the capstone is accepted against it, not against a demo.

In [ ]:
def bar() -> dict:
    """The acceptance bar for a multi-agent build, agreed before anyone runs anything."""
    generous = {"min_pass_rate": 0.6, "max_token_multiple": 10.0}   # anything can clear this
    defensible = {"min_pass_rate": 0.9,        # support answers are shown to customers
                  "max_token_multiple": 2.0}   # twice the cost needs a visible reason
    return defensible


def accepted(result: dict, b: dict) -> bool:
    """result: {"pass_rate": float, "token_multiple": float}. Given."""
    return result["pass_rate"] >= b["min_pass_rate"] and \
           result["token_multiple"] <= b["max_token_multiple"]

In [ ]:
# --- Self-check: Section 2
check("the bar is stricter than one that everything passes",
      lambda: bar()["min_pass_rate"] > 0.6 and bar()["max_token_multiple"] < 10.0,
      "a bar nothing can fail is not a bar")
check("Lab 1.4's team result would be rejected by it",
      lambda: not accepted({"pass_rate": 0.4, "token_multiple": 1.0}, bar()),
      "level on tokens, worse on answers -- cost cannot rescue that")
check("a genuinely better team would be accepted",
      lambda: accepted({"pass_rate": 1.0, "token_multiple": 1.8}, bar()))
score()

## Run it for real &mdash; let a model disagree with you

Describe one system in prose and ask for a recommendation. Then compare it with your rubric.

In [ ]:
DESCRIPTION = (
    "A support desk. A ticket arrives; the system reads it, looks up the runbook for its error "
    "code, and writes a reply to the customer. The steps are strictly sequential. We measured a "
    "three-agent version against a single agent on the same five cases: the single agent answered "
    "5 out of 5, the three-agent version 2 out of 5, for roughly the same token count.")

if llm_ready():
    def compare():
        opinion = ask("Should this be built as one agent or as multiple specialised agents? "
                      "Answer in two sentences.\n\n" + DESCRIPTION,
                      system="You are a pragmatic AI architect.")
        print("the model says:\n  ", opinion.strip()[:420])
        print("\nyour rubric says:", recommend(SYSTEMS["support desk (measured in 1.4)"]))
        print("\nOnly one of those two is checkable, and only one of them cites the measurement.")
    guard(compare)

### Read it

The model may well agree with you. That is not the point &mdash; ask it twice and see whether it
still does.

**A rubric is checkable and an opinion is not.** Yours is four lines, it names the facts it uses,
and anyone can run it on a system description and get the same answer. That is what makes it
usable in a design review, and it is why this is Module 1's take-home artifact rather than a slide.

**What you take from Module 1:** an agent is a loop you can write in fifteen lines; the four blocks
are real objects; tool descriptions are prompts and worth measuring; splitting into specialists has
a price you can measure; and the answer is sometimes &ldquo;no agent&rdquo;. Module 2 asks how the
agent should *think*, and Module 3 gives it a state you can pause and audit.

In [ ]:
score()

## Your turn

1. Add a fifth fact &mdash; `human_gate` &mdash; and decide whether it changes the recommendation
   or only the design. Getting that distinction right is most of Module 8.
2. Run your rubric over a system you actually own. If it says &ldquo;no agent&rdquo;, that is the
   most valuable answer it can give you.